In [1]:
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()

False

In [2]:
POSTGRES_HOST     = os.getenv('POSTGRES_HOST',     'postgres')
POSTGRES_PORT     = os.getenv('POSTGRES_PORT',     '5432')
POSTGRES_DB       = os.getenv('POSTGRES_DB',       'oil_pipeline')
POSTGRES_USER     = os.getenv('POSTGRES_USER',     'oil_user')
POSTGRES_PASSWORD = os.getenv('POSTGRES_PASSWORD', 'oil_password')

def get_pg_engine():
    return create_engine(
        f'postgresql+psycopg2://{POSTGRES_USER}:{POSTGRES_PASSWORD}'
        f'@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}'
    )

In [3]:
engine = get_pg_engine()

deliveries = pd.read_sql('SELECT * FROM deliveries', engine)
drivers    = pd.read_sql('SELECT * FROM drivers',    engine)
vehicles   = pd.read_sql('SELECT * FROM vehicles',   engine)

deliveries['date'] = pd.to_datetime(deliveries['date'])

print(f'deliveries: {deliveries.shape}')
print(f'drivers:    {drivers.shape}')
print(f'vehicles:   {vehicles.shape}')
deliveries.head(3)

deliveries: (30, 12)
drivers:    (5, 4)
vehicles:   (5, 4)


,delivery_id,date,source,destination,product_type,volume_ton,cost_usd,delay_hours,distance_km,weather_conditions,driver_id,vehicle_id
0,1,2025-10-01,Base-Khanty,Station-01,Diesel,32.5,2100.50,0.0,180.0,Clear,1,1
1,2,2025-10-01,Base-Tomsk,Station-02,Gasoline,28.0,1850.00,1.5,150.0,Rain,2,2
2,3,2025-10-01,Base-Tyumen,Station-03,Diesel,22.0,1650.25,0.0,120.0,Clear,3,3


In [4]:
df = deliveries.merge(drivers,  on='driver_id',  how='left') \
               .merge(vehicles, on='vehicle_id', how='left')

df.rename(columns={
    'name':           'driver_name',
    'experience_years': 'experience_years',
    'region':         'driver_region',
    'plate_number':   'plate_number',
    'capacity_ton':   'capacity_ton',
    'fuel_type':      'fuel_type',
}, inplace=True)

print(f'Итоговый датасет: {df.shape}')
print(f'NULL значения:\n{df.isnull().sum()}')
df.head(3)

Итоговый датасет: (30, 18)
NULL значения:
delivery_id           0
date                  0
source                0
destination           0
product_type          0
volume_ton            0
cost_usd              0
delay_hours           0
distance_km           0
weather_conditions    0
driver_id             0
vehicle_id            0
driver_name           0
experience_years      0
driver_region         0
plate_number          0
capacity_ton          0
fuel_type             0
dtype: int64


,delivery_id,date,source,destination,product_type,volume_ton,cost_usd,delay_hours,distance_km,weather_conditions,driver_id,vehicle_id,driver_name,experience_years,driver_region,plate_number,capacity_ton,fuel_type
0,1,2025-10-01,Base-Khanty,Station-01,Diesel,32.5,2100.50,0.0,180.0,Clear,1,1,Ivan Petrov,8,Khanty-Mansi,X123HM89,40.0,diesel
1,2,2025-10-01,Base-Tomsk,Station-02,Gasoline,28.0,1850.00,1.5,150.0,Rain,2,2,Sergey Sidorov,12,Tomsk,K456TM70,35.0,diesel
2,3,2025-10-01,Base-Tyumen,Station-03,Diesel,22.0,1650.25,0.0,120.0,Clear,3,3,Aleksey Smirnov,5,Tyumen,A789PO72,25.0,gas


In [5]:
df['cost_per_km'] = (df['cost_usd'] / df['distance_km']).round(3)

print('Cost per km по маршрутам (топ 10):')
print(df[['source','destination','distance_km','cost_usd','cost_per_km']]
      .sort_values('cost_per_km', ascending=False).head(10).to_string(index=False))

Cost per km по маршрутам (топ 10):
     source destination  distance_km  cost_usd  cost_per_km
  Base-Omsk  Station-14        100.0   1400.40       14.004
 Base-Tomsk  Station-08        140.0   1950.40       13.931
Base-Tyumen  Station-03        120.0   1650.25       13.752
  Base-Omsk  Station-26        104.0   1430.00       13.750
Base-Tyumen  Station-17        118.0   1620.20       13.731
 Base-Tomsk  Station-05        110.0   1500.00       13.636
Base-Tyumen  Station-21        122.0   1660.00       13.607
Base-Tyumen  Station-29        124.0   1680.00       13.548
  Base-Omsk  Station-18        105.0   1420.00       13.524
Base-Tyumen  Station-25        126.0   1700.00       13.492


In [6]:
delay_weather = df.groupby('weather_conditions').agg(
    avg_delay   = ('delay_hours', 'mean'),
    total_delay = ('delay_hours', 'sum'),
    trips       = ('delivery_id', 'count')
).round(2).sort_values('avg_delay', ascending=False).reset_index()

print('Задержки по погоде:')
print(delay_weather.to_string(index=False))
print()

df['distance_bin'] = pd.cut(df['distance_km'], bins=4)
delay_distance = df.groupby('distance_bin', observed=False).agg(
    avg_delay = ('delay_hours', 'mean'),
    trips     = ('delivery_id', 'count')
).round(2).reset_index()

print('Задержки по расстоянию:')
print(delay_distance.to_string(index=False))

Задержки по погоде:
weather_conditions  avg_delay  total_delay  trips
              Snow       3.50         10.5      3
              Rain       2.00         10.0      5
               Fog       1.50          4.5      3
            Cloudy       0.38          1.5      4
             Clear       0.10          1.5     15

Задержки по расстоянию:
  distance_bin  avg_delay  trips
(99.89, 127.5]       1.29     12
(127.5, 155.0]       0.64      7
(155.0, 182.5]       0.83      6
(182.5, 210.0]       0.60      5


In [7]:
driver_kpi = df.groupby(['driver_id', 'driver_name']).agg(
    total_trips      = ('delivery_id', 'count'),
    avg_delay        = ('delay_hours', 'mean'),
    total_delay      = ('delay_hours', 'sum'),
    avg_cost_per_km  = ('cost_per_km', 'mean'),
    total_volume_ton = ('volume_ton', 'sum'),
    total_cost_usd   = ('cost_usd', 'sum'),
    avg_distance_km  = ('distance_km', 'mean'),
).round(2).sort_values('avg_delay').reset_index()

print('KPI по водителям:')
driver_kpi

KPI по водителям:


,driver_id,driver_name,total_trips,avg_delay,total_delay,avg_cost_per_km,total_volume_ton,total_cost_usd,avg_distance_km
0,1,Ivan Petrov,5,0.20,1.0,11.73,168.4,11032.45,188.20
1,3,Aleksey Smirnov,7,0.43,3.0,13.26,165.1,11791.55,127.86
2,4,Pavel Egorov,4,0.62,2.5,11.75,130.7,8652.30,184.25
3,2,Sergey Sidorov,7,0.79,5.5,12.30,195.2,12930.40,150.43
4,5,Andrey Kuznetsov,7,2.29,16.0,13.59,141.1,10401.00,109.43


In [8]:
cost_weather = df.groupby('weather_conditions').agg(
    avg_cost_per_km = ('cost_per_km', 'mean'),
    avg_distance    = ('distance_km', 'mean'),
    trips           = ('delivery_id', 'count')
).round(2).sort_values('avg_cost_per_km', ascending=False).reset_index()

print('Стоимость по погодным условиям:')
print(cost_weather.to_string(index=False))
print()

cost_source = df.groupby('source').agg(
    avg_cost_per_km = ('cost_per_km', 'mean'),
    total_trips     = ('delivery_id', 'count'),
    avg_delay       = ('delay_hours', 'mean')
).round(2).sort_values('avg_cost_per_km', ascending=False).reset_index()

print('Стоимость по базам отправки:')
print(cost_source.to_string(index=False))

Стоимость по погодным условиям:
weather_conditions  avg_cost_per_km  avg_distance  trips
              Snow            13.58        105.67      3
            Cloudy            12.76        134.75      4
             Clear            12.65        153.87     15
              Rain            12.54        141.80      5
               Fog            11.79        173.00      3

Стоимость по базам отправки:
     source  avg_cost_per_km  total_trips  avg_delay
  Base-Omsk            13.58            6       2.58
Base-Tyumen            13.26            7       0.43
 Base-Tomsk            12.47            8       0.75
Base-Khanty            11.74            9       0.39


In [9]:
route_analysis = df.groupby(['source', 'destination']).agg(
    avg_cost_per_km = ('cost_per_km', 'mean'),
    avg_delay       = ('delay_hours', 'mean'),
    avg_distance    = ('distance_km', 'mean'),
    trips           = ('delivery_id', 'count')
).round(2).reset_index()

route_analysis['route_score'] = (
    route_analysis['avg_cost_per_km'] * 0.5 +
    route_analysis['avg_delay'] * 0.5
).round(3)

print('Маршруты отсортированные по эффективности (лучшие первые):')
print(route_analysis.sort_values('route_score').head(10).to_string(index=False))

Маршруты отсортированные по эффективности (лучшие первые):
     source destination  avg_cost_per_km  avg_delay  avg_distance  trips  route_score
Base-Khanty  Station-27            11.50        0.0         200.0      1        5.750
Base-Khanty  Station-01            11.67        0.0         180.0      1        5.835
Base-Khanty  Station-19            11.72        0.0         186.0      1        5.860
Base-Khanty  Station-09            11.77        0.0         170.0      1        5.885
Base-Khanty  Station-15            11.82        0.0         182.0      1        5.910
Base-Khanty  Station-06            11.90        0.0         185.0      1        5.950
 Base-Tomsk  Station-12            11.94        0.0         155.0      1        5.970
 Base-Tomsk  Station-24            12.08        0.0         149.0      1        6.040
 Base-Tomsk  Station-16            11.87        0.5         150.0      1        6.185
Base-Khanty  Station-23            12.00        0.5         175.0      1        6

In [10]:
mart_delay_weather = df[[
    'delivery_id', 'date', 'source', 'destination',
    'weather_conditions', 'delay_hours', 'distance_km',
    'cost_usd', 'cost_per_km', 'volume_ton'
]].copy()

mart_cost_distance = df[[
    'delivery_id', 'date', 'source', 'destination',
    'distance_km', 'cost_usd', 'cost_per_km',
    'volume_ton', 'weather_conditions', 'delay_hours'
]].copy()

mart_driver_kpi = driver_kpi.copy()

mart_routes = route_analysis.copy()

print(f'mart_delay_weather:  {mart_delay_weather.shape}')
print(f'mart_cost_distance:  {mart_cost_distance.shape}')
print(f'mart_driver_kpi:     {mart_driver_kpi.shape}')
print(f'mart_routes:         {mart_routes.shape}')

mart_delay_weather:  (30, 10)
mart_cost_distance:  (30, 10)
mart_driver_kpi:     (5, 9)
mart_routes:         (30, 7)


In [11]:
engine = get_pg_engine()

mart_delay_weather.to_sql('mart_delay_weather', engine, if_exists='replace', index=False)
mart_cost_distance.to_sql('mart_cost_distance', engine, if_exists='replace', index=False)
mart_driver_kpi.to_sql(   'mart_driver_kpi',    engine, if_exists='replace', index=False)
mart_routes.to_sql(       'mart_routes',        engine, if_exists='replace', index=False)

print('Все марты сохранены в PostgreSQL:')
print('  - mart_delay_weather')
print('  - mart_cost_distance')
print('  - mart_driver_kpi')
print('  - mart_routes')

Все марты сохранены в PostgreSQL:
  - mart_delay_weather
  - mart_cost_distance
  - mart_driver_kpi
  - mart_routes


In [12]:
print('=' * 55)
print('Delivery Analytics — Summary')
print('=' * 55)
print(f'Всего доставок:        {len(deliveries)}')
print(f'Водителей:             {len(drivers)}')
print(f'Транспортных средств:  {len(vehicles)}')
print()
print(f'Средняя задержка:      {df["delay_hours"].mean():.2f} ч')
print(f'Макс задержка:         {df["delay_hours"].max():.2f} ч')
print(f'Доставок без задержки: {(df["delay_hours"]==0).sum()}')
print()
print(f'Средний cost/km:       ${df["cost_per_km"].mean():.2f}')
print(f'Средняя дистанция:     {df["distance_km"].mean():.1f} км')
print()
print('Самая частая причина задержек:')
print(f'  {delay_weather.iloc[0]["weather_conditions"]} — '
      f'{delay_weather.iloc[0]["avg_delay"]:.2f} ч средняя задержка')
print()
print('Лучший водитель по задержкам:')
print(f'  {driver_kpi.iloc[0]["driver_name"]} — '
      f'{driver_kpi.iloc[0]["avg_delay"]:.2f} ч')
print('=' * 55)

Delivery Analytics — Summary
Всего доставок:        30
Водителей:             5
Транспортных средств:  5

Средняя задержка:      0.93 ч
Макс задержка:         4.00 ч
Доставок без задержки: 14

Средний cost/km:       $12.66
Средняя дистанция:     146.4 км

Самая частая причина задержек:
  Snow — 3.50 ч средняя задержка

Лучший водитель по задержкам:
  Ivan Petrov — 0.20 ч
